### High-Performance Pandas: eval() and query()

In [1]:
import numpy as np

In [17]:
rng = np.random.RandomState(42)
x = rng.rand(1000000)
y = rng.rand(1000000)
%timeit x + y

13.8 ms ± 1.52 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
mask = (x > 0.5) & (y < 0.5)

In [19]:
# roughly equivalent to:
tmp1 = (x > 0.5)
tmp2 = (y < 0.5)
mask = tmp1 & tmp2

In [ ]:
import numexpr 
mask_numexpr = numexpr.evaluate("(x > 0.5) & (y < 0.5)")
np.allclose(mask, mask_numexpr)

## pandas.eval() for Efficient Operations

In [22]:
import pandas as pd
nrows, ncols = 100000, 100
rng = np.random.RandomState(42)
df1, df2, df3, df4 = (pd.DataFrame(rng.rand(nrows, ncols))
for i in range(4))

In [23]:
%timeit df1 + df2 + df3 + df4 

372 ms ± 80.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [24]:
%timeit pd.eval("df1 + df2 + df3 + df4")

309 ms ± 22 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Operations supported by pd.eval()

In [27]:
df1, df2, df3, df4, df5 = (pd.DataFrame(rng.randint(0, 1000, (100, 3)))
for i in range(5))

In [ ]:
# Arithmetic operators
result1 = -df1 * df2 / (df3 + df4) - df5
result2 = pd.eval("-df1 * df2 / (df3 + df4) - df5")
np.allclose(result1, result2)

True

In [29]:
# Comparison operators 
result1 = (df1 < df2) & (df2 <= df3) & (df3 != df4)
result2 = pd.eval("df1 < df2 <= df3 != df4")
np.allclose(result1, result2)

True

In [30]:
# Bitwise operators
result1 = (df1 < 0.5) & (df2 < 0.5) | (df3 < df4)
result2 = pd.eval("(df1 < 0.5) & (df2 < 0.5) | (df3 < df4)")
np.allclose(result1, result2)

True

In [31]:
result3 = pd.eval("(df1 < 0.5) and (df2 < 0.5) or (df3 < df4)")
np.allclose(result1, result3)

True

In [32]:
# Object attributes and indices
result1 = df2.T[0] + df3.iloc[1]
result2 = pd.eval('df2.T[0] + df3.iloc[1]')
np.allclose(result1, result2)

True

### DataFrame.eval() for Column-Wise Operations

In [33]:
df = pd.DataFrame(rng.rand(1000, 3), columns=["A", "B", "C"])
df.head()

,A,B,C
0,0.061761,0.925463,0.997420
1,0.209863,0.280456,0.042148
2,0.738991,0.019046,0.715501
3,0.062857,0.516241,0.604588
4,0.204537,0.813392,0.244804


In [34]:
result1 = (df['A'] + df['B']) / (df['C'] - 1)
result2 = pd.eval("(df.A + df.B) / (df.C - 1)")
np.allclose(result1, result2)

True

In [36]:
result3 = df.eval('(A + B) / (C - 1)')
np.allclose(result1, result3)

True

In [37]:
df.head()

,A,B,C
0,0.061761,0.925463,0.997420
1,0.209863,0.280456,0.042148
2,0.738991,0.019046,0.715501
3,0.062857,0.516241,0.604588
4,0.204537,0.813392,0.244804


In [38]:
df.eval('D = (A + B) / C', inplace=True)
df.head()

,A,B,C,D
0,0.061761,0.925463,0.997420,0.989777
1,0.209863,0.280456,0.042148,11.633339
2,0.738991,0.019046,0.715501,1.059450
3,0.062857,0.516241,0.604588,0.957840
4,0.204537,0.813392,0.244804,4.158143


In [39]:
df.eval('D = (A - B) / C', inplace=True)
df.head()

,A,B,C,D
0,0.061761,0.925463,0.997420,-0.865935
1,0.209863,0.280456,0.042148,-1.674903
2,0.738991,0.019046,0.715501,1.006210
3,0.062857,0.516241,0.604588,-0.749906
4,0.204537,0.813392,0.244804,-2.487117


In [40]:
column_mean = df.mean(1)
result1 = df['A'] + column_mean
result2 = df.eval('A + @column_mean')
np.allclose(result1, result2)

True

### DataFrame.query() Method

In [41]:
result1 = df[(df.A < 0.5) & (df.B < 0.5)]
result2 = pd.eval('df[(df.A < 0.5) & (df.B < 0.5)]')
np.allclose(result1, result2)

True

In [42]:
result2 = df.query('A < 0.5 and B < 0.5')
np.allclose(result1, result2)

True

In [43]:
Cmean = df['C'].mean()
result1 = df[(df.A < Cmean) & (df.B < Cmean)]
result2 = df.query('A < @Cmean and B < @Cmean')
np.allclose(result1, result2)

True